In [ ]:
# Install required packages
%pip install -q transformers==4.35.0 torch==2.2.0 accelerate==0.24.0 bitsandbytes==0.41.1 huggingface_hub

# Import required packages
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
import os

# Check GPU availability
print("🔍 Checking GPU...")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU found! Please make sure you've selected GPU in Runtime > Change runtime type")


In [ ]:
# Handle HuggingFace authentication
print("\n🔐 Setting up HuggingFace authentication...")

# First try to get token from environment variable
hf_token = os.getenv('HF_TOKEN')

if not hf_token:
    print("Please enter your HuggingFace token:")
    print("(You can find it at: https://huggingface.co/settings/tokens)")
    hf_token = input("Token: ").strip()

# Login to HuggingFace
login(token=hf_token)
print("✅ Successfully logged in to HuggingFace!")


In [ ]:
# Load base model
print("🚀 Loading CodeLlama-7b-Instruct...")

model_name = "codellama/CodeLlama-7b-Instruct-hf"

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token,
    trust_remote_code=True
)

# Load model with 8-bit quantization
print("Loading model in 8-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    device_map="auto",
    load_in_8bit=True,  # 8-bit quantization works well on Colab GPUs
    torch_dtype=torch.float16,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()
print("✅ Model loaded successfully!")

# Print model info
print("\n📊 Model Information:")
print(f"Model name: {model_name}")
print(f"Model parameters: {model.num_parameters():,}")
print(f"GPU Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
# Define generation function
def generate_response(prompt, max_new_tokens=512):
    """Generate response from the model."""
    formatted_prompt = f"[INST] {prompt} [/INST]"
    inputs = tokenizer(formatted_prompt, return_tensors="pt", padding=True)
    
    # Move inputs to GPU if available
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
        
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response[len(formatted_prompt):].strip()

# Test with a simple FastAPI prompt
test_prompt = "Create a FastAPI GET endpoint that returns 'Hello, World!'"
print("Testing basic inference...")
print("Prompt:", test_prompt)
print("\nGenerated Response:")
print("-" * 40)
print(generate_response(test_prompt))
print("-" * 40)


In [ ]:
# Test more complex FastAPI prompts
complex_prompts = [
    "Create a FastAPI endpoint that accepts a JSON payload with 'name' and 'age' fields and returns a greeting",
    "Create a FastAPI endpoint that handles file upload and saves the file to disk",
    "Create a FastAPI endpoint that implements basic authentication using JWT tokens"
]

print("Testing complex prompts...")
for i, prompt in enumerate(complex_prompts, 1):
    print(f"\nTest {i}:")
    print("Prompt:", prompt)
    print("\nGenerated Response:")
    print("-" * 40)
    print(generate_response(prompt))
    print("-" * 40)


In [ ]:
# Set up OpenAI API key
from google.colab import userdata

# The API key should be added to Colab secrets with name 'OPENAI_API_KEY'
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise ValueError(
        "Please add your OpenAI API key to Colab secrets with name 'OPENAI_API_KEY'.\n"
        "1. Click on the key icon in the left sidebar\n"
        "2. Add a new secret with name 'OPENAI_API_KEY'\n"
        "3. Paste your OpenAI API key as the value\n"
        "4. Run this cell again"
    )

os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
print("✓ OpenAI API key loaded successfully")


In [ ]:
# Test cases for evaluation
test_cases = [
    {
        "name": "Dataset Example - Project List Endpoint",
        "prompt": "Create a FastAPI GET endpoint that lists projects for an organization. It should: 1) Accept organization_id as a parameter, 2) Use database session from dependencies, 3) Return a list of projects, 4) Include proper error handling if no projects found. Return ONLY the code without any explanation.",
    },
    {
        "name": "Simple Hello World",
        "prompt": "Create a FastAPI GET endpoint that returns 'Hello, World!'. Return ONLY the code without any explanation.",
    },
    {
        "name": "JSON Payload Handler",
        "prompt": "Create a FastAPI POST endpoint that accepts a JSON payload with 'name' and 'age' fields and returns a greeting. Return ONLY the code without any explanation.",
    },
    {
        "name": "Error Handling",
        "prompt": "Create a FastAPI endpoint that demonstrates proper error handling with HTTPException. Return ONLY the code without any explanation.",
    },
    {
        "name": "Path Parameters",
        "prompt": "Create a FastAPI endpoint that accepts a user_id as a path parameter and returns user information. Return ONLY the code without any explanation.",
    }
]


In [ ]:
import os
import openai
from typing import Dict, List, Any, Optional
from dataclasses import dataclass
import json

# Set up OpenAI API key
# Note: In Colab, you should set this as a secret
openai.api_key = os.getenv('OPENAI_API_KEY')

@dataclass
class FastAPIReviewCriteria:
    """Criteria for evaluating FastAPI code."""
    routing_patterns: bool = False  # Proper use of routes, HTTP methods
    type_hints: bool = False  # Python type hints and Pydantic models
    error_handling: bool = False  # HTTPException, status codes
    input_validation: bool = False  # Request models, query/path params
    response_models: bool = False  # Response schemas, status codes
    dependency_injection: bool = False  # Depends, proper DI patterns
    documentation: bool = False  # Docstrings, OpenAPI/Swagger
    best_practices: bool = False  # FastAPI conventions
    score: float = 0.0
    suggestions: List[str] = None
    detailed_feedback: str = ""

class GPTFastAPIJudge:
    """Uses GPT to evaluate FastAPI code quality."""
    
    def __init__(self):
        self.system_prompt = """You are an expert FastAPI code reviewer with deep knowledge of Python and REST APIs. 
Your task is to evaluate FastAPI code implementations focusing on:

1. Routing Patterns: Proper use of path operations, HTTP methods, and route organization
2. Type Safety: Python type hints and Pydantic models usage
3. Error Handling: Proper use of HTTPException and status codes
4. Input Validation: Request models, query parameters, path parameters
5. Response Models: Response schemas and status codes
6. Dependency Injection: Use of Depends and proper dependency patterns
7. Documentation: Docstrings and OpenAPI/Swagger documentation
8. FastAPI Best Practices: Following conventions and patterns

For each evaluation, provide:
1. A detailed analysis of each criterion
2. Specific code improvements
3. A score out of 100
4. Key strengths and weaknesses

Format your response exactly as:
DETAILED ANALYSIS:
[Your analysis of each criterion]

SCORE: X/100

STRENGTHS:
- [Strength 1]
- [Strength 2]
...

WEAKNESSES:
- [Weakness 1]
- [Weakness 2]
...

SUGGESTED IMPROVEMENTS:
1. [Improvement 1]
2. [Improvement 2]
...
"""

    def evaluate(self, prompt: str, code: str) -> FastAPIReviewCriteria:
        """Evaluates FastAPI code using GPT."""
        try:
            # Prepare the evaluation request
            messages = [
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": f"""Please evaluate this FastAPI code implementation.

Original Task:
{prompt}

Code to Review:
```python
{code}
```"""}
            ]

            # Get GPT's evaluation
            response = openai.ChatCompletion.create(
                model="gpt-4",  # Using GPT-4 for best results
                messages=messages,
                temperature=0.3,  # Lower temperature for more consistent evaluation
                max_tokens=2000
            )

            # Parse the response
            evaluation_text = response.choices[0].message.content
            
            # Initialize result
            result = FastAPIReviewCriteria()
            result.suggestions = []
            
            # Parse sections
            sections = {
                "DETAILED ANALYSIS:": "",
                "SCORE:": "",
                "STRENGTHS:": [],
                "WEAKNESSES:": [],
                "SUGGESTED IMPROVEMENTS:": []
            }
            
            current_section = None
            for line in evaluation_text.split("\n"):
                line = line.strip()
                if line in sections:
                    current_section = line
                    continue
                if current_section and line:
                    if isinstance(sections[current_section], list):
                        if line.startswith("-") or line.startswith("1."):
                            sections[current_section].append(line.lstrip("- 123456789.").strip())
                    else:
                        sections[current_section] += line + "\n"
            
            # Extract score
            score_text = sections["SCORE:"].strip()
            if "/" in score_text:
                result.score = float(score_text.split("/")[0])
            
            # Set detailed feedback
            result.detailed_feedback = sections["DETAILED ANALYSIS:"].strip()
            
            # Set suggestions
            result.suggestions = sections["SUGGESTED IMPROVEMENTS:"]
            
            # Set criteria based on analysis
            result.routing_patterns = "routing" in result.detailed_feedback.lower()
            result.type_hints = "type" in result.detailed_feedback.lower()
            result.error_handling = "error" in result.detailed_feedback.lower()
            result.input_validation = "input" in result.detailed_feedback.lower()
            result.response_models = "response" in result.detailed_feedback.lower()
            result.dependency_injection = "depend" in result.detailed_feedback.lower()
            result.documentation = "doc" in result.detailed_feedback.lower()
            result.best_practices = "practice" in result.detailed_feedback.lower()
            
            return result
            
        except Exception as e:
            # Handle any API errors
            result = FastAPIReviewCriteria()
            result.detailed_feedback = f"Evaluation failed: {str(e)}"
            result.score = 0
            result.suggestions = ["Could not complete evaluation"]
            return result

def format_gpt_evaluation(result: FastAPIReviewCriteria) -> str:
    """Formats the GPT evaluation result into a readable string."""
    output = []
    output.append("🤖 GPT FastAPI Code Review")
    output.append("=" * 40)
    
    # Score
    output.append(f"📊 Overall Score: {result.score}/100")
    
    # Criteria Check
    output.append("\n✨ Criteria Check:")
    criteria = {
        "Routing Patterns": result.routing_patterns,
        "Type Hints": result.type_hints,
        "Error Handling": result.error_handling,
        "Input Validation": result.input_validation,
        "Response Models": result.response_models,
        "Dependency Injection": result.dependency_injection,
        "Documentation": result.documentation,
        "Best Practices": result.best_practices
    }
    for criterion, present in criteria.items():
        output.append(f"{'✓' if present else '✗'} {criterion}")
    
    # Detailed Feedback
    if result.detailed_feedback:
        output.append("\n📝 Detailed Analysis:")
        output.append(result.detailed_feedback)
    
    # Suggestions
    if result.suggestions:
        output.append("\n💡 Suggested Improvements:")
        for i, suggestion in enumerate(result.suggestions, 1):
            output.append(f"{i}. {suggestion}")
    
    return "\n".join(output)


In [ ]:
# Initialize GPT-based FastAPI judge
gpt_judge = GPTFastAPIJudge()

print("🧪 Running GPT-based FastAPI Code Evaluation\n")

for test_case in test_cases:
    print("=" * 60)
    print(f"Test: {test_case['name']}")
    print(f"Prompt: {test_case['prompt']}")
    print("=" * 60 + "\n")
    
    # Generate response
    response = generate_text(
        model=model,
        tokenizer=tokenizer,
        prompt=test_case['prompt'],
        max_length=512
    )
    
    print("Generated Code:")
    print("-" * 40)
    print(response)
    print("-" * 40 + "\n")
    
    # Run GPT evaluation
    print("GPT Evaluation:")
    code = extract_code_from_markdown(response)
    evaluation = gpt_judge.evaluate(test_case['prompt'], code)
    print(format_gpt_evaluation(evaluation))
    print("\n" + "=" * 60 + "\n")


In [ ]:
# FastAPI Evaluator Implementation
import ast
import re
from typing import List, Dict, Any, Optional, Set
from dataclasses import dataclass, field
import json
from enum import Enum

class FastAPIBestPractices(Enum):
    """Enum for FastAPI best practices patterns"""
    PYDANTIC_MODELS = r"class\s+\w+(\w*Model|\w*Schema|\w*Request|\w*Response)"
    RESPONSE_MODEL = r"@\w+\.\w+\([^)]*response_model\s*="
    STATUS_CODES = r"(status_code\s*=|status\.\w+)"
    DEPENDENCIES = r"Depends\([^)]+\)"
    ASYNC_DEF = r"async\s+def"
    PATH_PARAMS = r"{[^}]+}"
    QUERY_PARAMS = r"(\w+\s*:\s*Optional|\w+\s*=\s*Query\()"
    BODY_MODELS = r"(\w+\s*:\s*\w+Model|\w+\s*:\s*\w+Schema)"

def extract_code_from_markdown(text: str) -> str:
    """Extract Python code from markdown-formatted text with code blocks."""
    # Find content between triple backticks
    code_blocks = re.findall(r'```(?:python)?(.*?)```', text, re.DOTALL)
    if code_blocks:
        # Return the first code block found
        return code_blocks[0].strip()
    # If no code blocks found, try to find the code directly
    # Remove any explanatory text that comes before or after the code
    if 'from fastapi import' in text:
        # Find the first import statement and everything after it
        code = text[text.find('from fastapi import'):]
        # Remove any explanatory text that might come after the code
        if 'This endpoint' in code:
            code = code[:code.find('This endpoint')]
        return code.strip()
    return text.strip()

@dataclass
class EvaluationResult:
    """Stores the evaluation results for a single test case."""
    prompt: str
    response: str
    is_valid_python: bool = False
    has_imports: bool = False
    has_router: bool = False
    has_endpoint: bool = False
    has_type_hints: bool = False
    has_docstring: bool = False
    has_error_handling: bool = False
    has_pydantic_models: bool = False
    has_response_model: bool = False
    has_status_codes: bool = False
    has_dependencies: bool = False
    has_async_def: bool = False
    has_path_params: bool = False
    has_query_params: bool = False
    has_body_models: bool = False
    required_imports: Set[str] = field(default_factory=set)
    missing_imports: Set[str] = field(default_factory=set)
    extracted_endpoints: List[Dict[str, Any]] = field(default_factory=list)
    score: float = 0.0
    error_message: Optional[str] = None
    
    def calculate_score(self) -> float:
        """Calculate weighted score based on various criteria."""
        weights = {
            'is_valid_python': 1.0,
            'has_imports': 0.8,
            'has_router': 0.8,
            'has_endpoint': 1.0,
            'has_type_hints': 0.7,
            'has_docstring': 0.3,  # Less critical
            'has_error_handling': 0.9,
            'has_pydantic_models': 0.6,
            'has_response_model': 0.6,
            'has_status_codes': 0.7,
            'has_dependencies': 0.5,
            'has_async_def': 0.3,  # Optional
            'has_path_params': 0.4,
            'has_query_params': 0.4,
            'has_body_models': 0.5
        }
        
        total_weight = sum(weights.values())
        weighted_sum = sum(
            weights[attr] * getattr(self, attr)
            for attr in weights.keys()
        )
        
        return weighted_sum / total_weight

class FastAPIEvaluator:
    """Evaluates FastAPI code generation responses."""
    
    def __init__(self):
        # Common FastAPI imports and their patterns
        self.import_patterns = {
            "fastapi": [r"from\s+fastapi\s+import", r"import\s+fastapi"],
            "FastAPI": [r"FastAPI"],
            "APIRouter": [r"APIRouter"],
            "HTTPException": [r"HTTPException"],
            "status": [r"status\.HTTP_[0-9]+"],
            "Response": [r"Response"],
            "Request": [r"Request"],
            "Depends": [r"Depends"],
            "Body": [r"Body"],
            "Query": [r"Query"],
            "Path": [r"Path"],
            "BaseModel": [r"BaseModel"],
        }
    
    def evaluate_response(self, prompt: str, response: str) -> EvaluationResult:
        """Evaluates a single response."""
        # First extract actual code from the response
        code = extract_code_from_markdown(response)
        result = EvaluationResult(prompt=prompt, response=code)
        
        try:
            # Check if it's valid Python code
            ast.parse(code)
            result.is_valid_python = True
        except SyntaxError as e:
            result.error_message = f"Invalid Python syntax: {str(e)}"
            return result
        
        # Check for imports and track them
        for import_name, patterns in self.import_patterns.items():
            if any(re.search(pattern, code) for pattern in patterns):
                result.required_imports.add(import_name)
            else:
                result.missing_imports.add(import_name)
        result.has_imports = len(result.required_imports) > 0
        
        # Check for router/app initialization
        result.has_router = bool(re.search(r"(app\s*=\s*FastAPI\(\)|router\s*=\s*APIRouter\(\))", code))
        
        # Check for endpoints with improved pattern
        endpoint_pattern = r"@\s*(app|router)\.(get|post|put|delete|patch)\s*\(\s*['\"]([^'\"]+)['\"]\s*[,)]"
        endpoints = re.finditer(endpoint_pattern, code)
        result.extracted_endpoints = []
        
        for match in endpoints:
            decorator, method, path = match.groups()
            result.extracted_endpoints.append({
                "decorator": decorator,
                "method": method.upper(),
                "path": path
            })
        
        result.has_endpoint = len(result.extracted_endpoints) > 0
        
        # Check for type hints with improved pattern
        type_hint_pattern = r"def\s+\w+\s*\([^)]*:\s*\w+[\[\],\s]*\w*"
        result.has_type_hints = bool(re.search(type_hint_pattern, code))
        
        # Check for docstrings with improved pattern
        docstring_pattern = r'("""[\s\S]*?"""|\'\'\'[\s\S]*?\'\'\')'
        result.has_docstring = bool(re.search(docstring_pattern, code))
        
        # Enhanced error handling detection
        error_patterns = [
            r"HTTPException",
            r"try\s*:",
            r"raise\s+\w+",
            r"status\.HTTP_[45]\d\d",  # 4xx and 5xx status codes
            r"status_code\s*=\s*[45]\d\d"
        ]
        result.has_error_handling = any(bool(re.search(pattern, code)) for pattern in error_patterns)
        
        # Check FastAPI best practices
        for practice in FastAPIBestPractices:
            attr_name = f"has_{practice.name.lower()}"
            if hasattr(result, attr_name):
                setattr(result, attr_name, bool(re.search(practice.value, code)))
        
        # Calculate weighted score
        result.score = result.calculate_score()
        
        return result
    
    def format_evaluation_result(self, result: EvaluationResult) -> str:
        """Formats the evaluation result into a readable string."""
        output = []
        output.append("📊 Evaluation Results:")
        
        # Basic checks
        output.append("\n🔍 Basic Checks:")
        output.append(f"✓ Valid Python: {result.is_valid_python}")
        if result.error_message:
            output.append(f"⚠️ Error: {result.error_message}")
        output.append(f"✓ Has Router/App: {result.has_router}")
        
        # Import analysis
        output.append("\n📦 Import Analysis:")
        output.append(f"✓ Has Required Imports: {result.has_imports}")
        if result.required_imports:
            output.append("  Found imports:")
            for imp in sorted(result.required_imports):
                output.append(f"    • {imp}")
        if result.missing_imports:
            output.append("  Missing recommended imports:")
            for imp in sorted(result.missing_imports):
                output.append(f"    • {imp}")
        
        # Endpoint analysis
        output.append("\n🛣️ Endpoint Analysis:")
        output.append(f"✓ Has Endpoints: {result.has_endpoint}")
        if result.extracted_endpoints:
            output.append("  Endpoints found:")
            for endpoint in result.extracted_endpoints:
                output.append(f"    • {endpoint['method']} {endpoint['path']}")
        
        # Code quality
        output.append("\n📝 Code Quality:")
        output.append(f"✓ Has Type Hints: {result.has_type_hints}")
        output.append(f"✓ Has Docstrings: {result.has_docstring}")
        output.append(f"✓ Has Error Handling: {result.has_error_handling}")
        
        # FastAPI best practices
        output.append("\n✨ FastAPI Best Practices:")
        output.append(f"✓ Uses Pydantic Models: {result.has_pydantic_models}")
        output.append(f"✓ Specifies Response Models: {result.has_response_model}")
        output.append(f"✓ Uses Status Codes: {result.has_status_codes}")
        output.append(f"✓ Uses Dependencies: {result.has_dependencies}")
        output.append(f"✓ Uses Async Functions: {result.has_async_def}")
        output.append(f"✓ Has Path Parameters: {result.has_path_params}")
        output.append(f"✓ Has Query Parameters: {result.has_query_params}")
        output.append(f"✓ Uses Request/Response Models: {result.has_body_models}")
        
        # Final score
        output.append(f"\n🎯 Overall Score: {result.score:.2%}")
        
        return "\n".join(output)

# Initialize evaluator
evaluator = FastAPIEvaluator()


In [ ]:
# Test cases for evaluation
test_cases = [
    {
        "name": "Dataset Example - Project List Endpoint",
        "prompt": "Create a FastAPI GET endpoint that lists projects for an organization. It should: 1) Accept organization_id as a parameter, 2) Use database session from dependencies, 3) Return a list of projects, 4) Include proper error handling if no projects found. Return ONLY the code without any explanation."
    },
    {
        "name": "Simple Hello World",
        "prompt": "Create a FastAPI GET endpoint that returns 'Hello, World!'. Return ONLY the code without any explanation."
    },
    {
        "name": "JSON Payload Handler",
        "prompt": "Create a FastAPI POST endpoint that accepts a JSON payload with 'name' and 'age' fields and returns a greeting. Return ONLY the code without any explanation."
    },
    {
        "name": "Error Handling",
        "prompt": "Create a FastAPI endpoint that demonstrates proper error handling with HTTPException. Return ONLY the code without any explanation."
    },
    {
        "name": "Path Parameters",
        "prompt": "Create a FastAPI endpoint that accepts a user_id as a path parameter and returns user information. Return ONLY the code without any explanation."
    }
]

# Run evaluations
print("🧪 Running FastAPI Code Generation Tests\n")

all_scores = []
for test in test_cases:
    print(f"\n{'='*60}")
    print(f"Test: {test['name']}")
    print(f"Prompt: {test['prompt']}")
    print(f"{'='*60}\n")
    
    # Generate response
    response = generate_response(test['prompt'])
    print("Generated Code:")
    print("-" * 40)
    print(response)
    print("-" * 40)
    
    # Evaluate response
    result = evaluator.evaluate_response(test['prompt'], response)
    print("\nEvaluation:")
    print(evaluator.format_evaluation_result(result))
    all_scores.append(result.score)

# Print summary
print("\n📈 Overall Results:")
print(f"Average Score: {sum(all_scores)/len(all_scores):.2%}")
print(f"Best Score: {max(all_scores):.2%}")
print(f"Worst Score: {min(all_scores):.2%}")
